# Zone Counting: Market Square

This notebook counts how many people are inside different areas of a busy square, frame by frame. It uses:

1. [RF-DETR](https://github.com/roboflow/rf-detr) with tiled inference to detect people in a 4K frame.
2. Supervision's `sv.PolygonZone` to check which detections fall inside each area.
3. A time series of occupancy per zone, plotted at the end.

Unlike line crossing, zone occupancy is answered frame by frame ("how many people are here right now?"), so we don't need a tracker.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import supervision as sv
from rfdetr import RFDETRNano
from rfdetr.assets.coco_classes import COCO_CLASSES
from supervision.assets import VideoAssets, download_assets
from tqdm.auto import tqdm

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

## Download the video

`MARKET_SQUARE` is a portrait 4K clip (2160×3840, 60 fps) filmed from a fixed position above a square with a fountain. A fixed camera matters: zones are defined in pixel coordinates, so they only stay on the right spot if the camera doesn't move.

In [ ]:
SOURCE_VIDEO_PATH = download_assets(VideoAssets.MARKET_SQUARE, directory=DATA_DIR)
TARGET_VIDEO_PATH = OUTPUT_DIR / "market_square_zones.mp4"

video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
video_info

In [ ]:
frame = next(sv.get_video_frames_generator(SOURCE_VIDEO_PATH))
sv.plot_image(frame, size=(8, 14))

## Detect people

Running the model on the full frame barely works here: the tall 4K frame is shrunk to 384 px for Nano, and people become a few pixels wide. As in the people-walking notebook, `sv.InferenceSlicer` runs the model on 640×640 tiles instead.

Two slicer settings matter in a crowd:

- **`batch_size`** sends several tiles to the model at once, which is noticeably faster than one tile at a time.
- **`overlap_metric=IOS`** (intersection over the *smaller* box) merges duplicates across tile borders. A person cut by a tile edge produces a partial box inside a full box; their IoU is low, so the default IoU check keeps both, but IoS is high, so one gets removed.

In [ ]:
model = RFDETRNano()
PERSON_CLASS_ID = next(k for k, v in COCO_CLASSES.items() if v == "person")


def detect_batch(images):
    return model.predict([image[:, :, ::-1].copy() for image in images], threshold=0.3)


slicer = sv.InferenceSlicer(
    callback=detect_batch,
    slice_wh=640,
    overlap_wh=128,
    batch_size=8,
    overlap_metric=sv.OverlapMetric.IOS,
)


def detect_people(frame):
    detections = slicer(frame)
    return detections[(detections.class_id == PERSON_CLASS_ID) & (detections.confidence >= 0.5)]

In [ ]:
full_frame = model.predict(frame[:, :, ::-1].copy(), threshold=0.5)
full_frame = full_frame[full_frame.class_id == PERSON_CLASS_ID]
detections = detect_people(frame)

print(f"Full frame: {len(full_frame)} people")
print(f"Sliced:     {len(detections)} people")

## Define the zones

Each zone is a polygon in pixel coordinates. A handy way to find them is to plot a frame with a grid, or use Roboflow's [PolygonZone tool](https://polygonzone.roboflow.com) to click the points on an image.

We split the scene into three areas:

- **fountain**: the fountain and the space around it, where people stop and sit.
- **square**: the patterned middle of the square.
- **street**: the plain pavement at the bottom.

By default a detection belongs to a zone when its bottom-centre point (the person's feet) is inside the polygon, so each person is counted in at most one zone.

In [ ]:
square_edge = [(2160, 2960), (1900, 3040), (1500, 3120), (1000, 3140), (500, 3090), (0, 2990)]

ZONE_POLYGONS = {
    "fountain": np.array([(800, 1020), (1400, 1020), (1620, 1220), (1620, 1560), (1420, 1720), (780, 1720), (580, 1560), (580, 1220)]),
    "square": np.array([(0, 1800), (2160, 1800), *square_edge]),
    "street": np.array([*square_edge[::-1], (2160, 3840), (0, 3840)]),
}
ZONE_COLORS = sv.ColorPalette.from_hex(["#3b82f6", "#22c55e", "#f97316"])

zones = {name: sv.PolygonZone(polygon=polygon) for name, polygon in ZONE_POLYGONS.items()}

Annotators are created per zone so each one can have its own colour. People outside every zone are drawn in grey.

In [ ]:
text_scale = sv.calculate_optimal_text_scale(video_info.resolution_wh)
thickness = sv.calculate_optimal_line_thickness(video_info.resolution_wh)

zone_annotators = {
    name: sv.PolygonZoneAnnotator(
        zone=zone,
        color=ZONE_COLORS.by_idx(i),
        thickness=thickness,
        text_scale=text_scale,
        text_thickness=thickness,
        opacity=0.2,
    )
    for i, (name, zone) in enumerate(zones.items())
}
box_annotators = {
    name: sv.BoxAnnotator(color=ZONE_COLORS.by_idx(i), thickness=thickness) for i, name in enumerate(zones)
}
outside_annotator = sv.BoxAnnotator(color=sv.Color.from_hex("#9ca3af"), thickness=thickness)


def annotate(frame, detections):
    annotated = frame.copy()
    outside = np.ones(len(detections), dtype=bool)
    counts = {}

    for name, zone in zones.items():
        in_zone = zone.trigger(detections)
        outside &= ~in_zone
        counts[name] = int(in_zone.sum())
        annotated = zone_annotators[name].annotate(annotated, label=f"{name}: {counts[name]}")
        annotated = box_annotators[name].annotate(annotated, detections[in_zone])

    annotated = outside_annotator.annotate(annotated, detections[outside])
    return annotated, counts

In [ ]:
annotated_frame, counts = annotate(frame, detections)
print(counts)
sv.plot_image(annotated_frame, size=(8, 14))

## Process the video

Tiled inference on a 4K frame takes around half a second, and at 60 fps the video has far more frames than we need to measure occupancy. We process every third frame (`stride=3`) and write the output at 20 fps, so it still plays at real-time speed. The whole clip takes about a minute and a half on an Apple Silicon Mac.

Instead of `sv.process_video` we write the loop ourselves with `sv.VideoSink`, which lets us skip frames and collect the counts as we go.

In [ ]:
STRIDE = 3
output_info = sv.VideoInfo(width=video_info.width, height=video_info.height, fps=video_info.fps / STRIDE)
frames = sv.get_video_frames_generator(SOURCE_VIDEO_PATH, stride=STRIDE)

records = []
with sv.VideoSink(str(TARGET_VIDEO_PATH), output_info) as sink:
    for index, frame in enumerate(tqdm(frames, total=video_info.total_frames // STRIDE + 1)):
        detections = detect_people(frame)
        annotated, counts = annotate(frame, detections)
        sink.write_frame(annotated)
        records.append({"time_s": index * STRIDE / video_info.fps, "total": len(detections), **counts})

## Results

Occupancy per zone over time, plus summary statistics.

In [ ]:
occupancy = pd.DataFrame(records).set_index("time_s")
occupancy.describe().loc[["mean", "min", "max"]].round(1)

In [ ]:
ax = occupancy[list(zones)].plot(
    figsize=(12, 4), color=[ZONE_COLORS.by_idx(i).as_hex() for i in range(len(zones))]
)
ax.set_xlabel("time (s)")
ax.set_ylabel("people in zone")
ax.set_title("Zone occupancy over time");

In [ ]:
samples = list(sv.get_video_frames_generator(TARGET_VIDEO_PATH, stride=len(records) // 3))[:3]
sv.plot_images_grid(samples, grid_size=(1, 3), size=(16, 10))

Open `output/market_square_zones.mp4` to watch the full video.

**Things to try**

- Draw your own zones, e.g. the café terrace at the top, or split the square into left and right halves.
- Change `triggering_anchors` on `sv.PolygonZone` to `sv.Position.CENTER` and see how people near the zone borders are assigned.
- Add a tracker (as in the previous notebooks) to measure *how long* each person stays in a zone instead of just how many are there.